# MFCC Feature Extraction for Beehive Acoustic Monitoring

## Aim
This notebook extracts **Mel-Frequency Cepstral Coefficients (MFCCs)** from preprocessed beehive recordings, aligns the acoustic feature timeline with environmental logs, and visualises temporal variation across the session.

In this project, MFCCs are used as a compact representation of the hive sound spectrum. They are useful for tracking changes in:
- collective buzzing texture
- spectral shape variation over time
- possible behavioural or environmental shifts reflected in the acoustic profile

Because the recordings were already normalised during preprocessing, the MFCCs here should be interpreted as descriptors of **relative spectral variation**, not as absolute calibrated sound measurements.

## 1. Import libraries

In [ ]:
import os
from glob import glob
from datetime import timedelta

import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

## 2. Configuration

Update the paths and session metadata below for each dataset. Keeping all settings together makes the notebook easier to reuse, and more transparent for dissertation documentation.

In [ ]:
# =========================
# USER CONFIGURATION
# =========================

# Audio and metadata paths
INPUT_FOLDER = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024"
ENV_PATH = r"X:\Dissertation Files\Set I\Preprocessed files Set I.II 13.08.2024 - 15.08.2024\Env_logs\HaniBi_20240815T134229+0200.xlsx"

# Output folders
MFCC_OUTPUT_FOLDER = os.path.join(INPUT_FOLDER, "MFCC_features")
PLOT_OUTPUT_FOLDER = os.path.join(MFCC_OUTPUT_FOLDER, "chunk_plots")
os.makedirs(MFCC_OUTPUT_FOLDER, exist_ok=True)
os.makedirs(PLOT_OUTPUT_FOLDER, exist_ok=True)

# Session timing
BASE_START_TIME = pd.Timestamp("2024-08-13 11:20:00")
START_SLICE = pd.Timestamp("2024-08-13 11:20:00")
END_SLICE = pd.Timestamp("2024-08-15 13:42:29")

# MFCC extraction parameters
N_MFCC = 13
N_FFT = 2048
HOP_LENGTH = 16000

# Plotting parameters
CHUNK_SECONDS = 15 * 60
PLOT_MFCCS = ["mfcc_1", "mfcc_2", "mfcc_3"]  # change here if you want to inspect other coefficients

## 3. Define helper functions

The extraction workflow is written as reusable functions so the notebook is easier to validate, maintain, and later convert into a standalone Python module if needed.

In [ ]:
def extract_mfcc_with_timestamps(audio_path, start_time, n_mfcc=13, n_fft=2048, hop_length=16000):
    """
    Extract MFCCs from a WAV file and return a timestamped DataFrame.

    Parameters
    ----------
    audio_path : str
        Path to the audio file.
    start_time : pandas.Timestamp
        Real-world start time assigned to the beginning of the file.
    n_mfcc : int
        Number of MFCC coefficients to extract.
    n_fft : int
        FFT window size used in MFCC calculation.
    hop_length : int
        Hop size between successive frames.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing timestamped MFCC features.
    """
    y, sr = librosa.load(audio_path, sr=None)

    mfccs = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop_length
    )

    frame_times = librosa.frames_to_time(
        np.arange(mfccs.shape[1]),
        sr=sr,
        hop_length=hop_length
    )
    timestamps = [start_time + timedelta(seconds=float(t)) for t in frame_times]

    mfcc_df = pd.DataFrame(
        mfccs.T,
        columns=[f"mfcc_{i+1}" for i in range(n_mfcc)]
    )
    mfcc_df["timestamp"] = timestamps
    mfcc_df["source_file"] = os.path.basename(audio_path)

    return mfcc_df


def load_and_prepare_environmental_log(env_path, start_slice=None, end_slice=None):
    """
    Load the environmental spreadsheet, standardise the timestamp column,
    remove duplicate timestamps, and optionally slice to the recording window.
    """
    env_df = pd.read_excel(env_path)

    if "Date" in env_df.columns:
        env_df = env_df.rename(columns={"Date": "timestamp"})
    elif "timestamp" not in env_df.columns:
        raise ValueError("Environmental log must contain a 'Date' or 'timestamp' column.")

    env_df["timestamp"] = pd.to_datetime(env_df["timestamp"])
    env_df = env_df.drop_duplicates(subset="timestamp").sort_values("timestamp").reset_index(drop=True)

    if start_slice is not None:
        env_df = env_df[env_df["timestamp"] >= start_slice]
    if end_slice is not None:
        env_df = env_df[env_df["timestamp"] <= end_slice]

    return env_df.reset_index(drop=True)


def compute_sequential_start_times(file_list, base_start_time):
    """
    Assign a real-world start time to each WAV file using cumulative duration.
    """
    start_times = []
    durations = []
    elapsed = 0.0

    for file_path in file_list:
        duration = librosa.get_duration(path=file_path)
        start_times.append(base_start_time + timedelta(seconds=elapsed))
        durations.append(duration)
        elapsed += duration

    return start_times, durations


def align_mfcc_with_environmental_log(mfcc_df, env_df):
    """
    Align MFCC timestamps with environmental measurements using nearest-neighbour merge.
    """
    aligned_df = pd.merge_asof(
        mfcc_df.sort_values("timestamp"),
        env_df.sort_values("timestamp"),
        on="timestamp",
        direction="nearest"
    )
    return aligned_df


def normalise_columns(df, columns):
    """
    Min-max normalise selected columns for comparative plotting.
    A new DataFrame is returned, preserving the original columns.
    """
    norm_df = df.copy()
    scaler = MinMaxScaler()

    valid_columns = [col for col in columns if col in norm_df.columns]
    norm_values = scaler.fit_transform(norm_df[valid_columns])

    for idx, col in enumerate(valid_columns):
        norm_df[f"{col}_norm"] = norm_values[:, idx]

    return norm_df


def plot_mfcc_overview(norm_df, mfcc_columns, env_columns=None):
    """
    Plot selected normalised MFCC trajectories across the full session.
    """
    plt.figure(figsize=(14, 5))

    for col in mfcc_columns:
        plt.plot(norm_df["timestamp"], norm_df[f"{col}_norm"], label=f"{col} (normalised)")

    if env_columns:
        for col in env_columns:
            norm_col = f"{col}_norm"
            if norm_col in norm_df.columns:
                plt.plot(norm_df["timestamp"], norm_df[norm_col], label=f"{col} (normalised)")

    plt.title("Normalised MFCC Trends Over Full Session")
    plt.xlabel("Timestamp")
    plt.ylabel("Normalised Value")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_mfcc_chunks(norm_df, mfcc_columns, output_dir, chunk_seconds=900, env_columns=None):
    """
    Plot selected MFCC coefficients in smaller time chunks and save each figure.
    """
    start_time = norm_df["timestamp"].min()
    end_time = norm_df["timestamp"].max()

    current_start = start_time
    chunk_counter = 1

    while current_start < end_time:
        current_end = current_start + pd.Timedelta(seconds=chunk_seconds)
        chunk_df = norm_df[(norm_df["timestamp"] >= current_start) & (norm_df["timestamp"] < current_end)]

        if not chunk_df.empty:
            plt.figure(figsize=(14, 5))

            for col in mfcc_columns:
                plt.plot(chunk_df["timestamp"], chunk_df[f"{col}_norm"], label=f"{col} (normalised)")

            if env_columns:
                for col in env_columns:
                    norm_col = f"{col}_norm"
                    if norm_col in chunk_df.columns:
                        plt.plot(chunk_df["timestamp"], chunk_df[norm_col], label=f"{col} (normalised)")

            plt.title(f"MFCC Trends | Chunk {chunk_counter}")
            plt.xlabel("Timestamp")
            plt.ylabel("Normalised Value")
            plt.legend()
            plt.tight_layout()

            plot_name = f"mfcc_chunk_{chunk_counter:03d}.png"
            plt.savefig(os.path.join(output_dir, plot_name), dpi=300, bbox_inches="tight")
            plt.show()

        current_start = current_end
        chunk_counter += 1

## 4. Load and inspect environmental data

In [ ]:
env_df = load_and_prepare_environmental_log(
    env_path=ENV_PATH,
    start_slice=START_SLICE,
    end_slice=END_SLICE
)

env_df.head()

## 5. Batch MFCC extraction across all preprocessed recordings

This step processes every WAV file in the session folder, calculates MFCCs, assigns real timestamps using cumulative duration, and saves one CSV file per recording.

In [ ]:
file_list = sorted(glob(os.path.join(INPUT_FOLDER, "*.wav")))
start_times, durations = compute_sequential_start_times(file_list, BASE_START_TIME)

processing_log = []

for file_path, file_start_time, duration in zip(file_list, start_times, durations):
    file_name = os.path.basename(file_path)

    try:
        mfcc_df = extract_mfcc_with_timestamps(
            audio_path=file_path,
            start_time=file_start_time,
            n_mfcc=N_MFCC,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        output_name = os.path.splitext(file_name)[0] + "_mfcc.csv"
        output_path = os.path.join(MFCC_OUTPUT_FOLDER, output_name)
        mfcc_df.to_csv(output_path, index=False)

        processing_log.append({
            "file": file_name,
            "status": "success",
            "start_time": file_start_time,
            "duration_seconds": duration,
            "rows_written": len(mfcc_df),
            "output_file": output_name
        })

    except Exception as e:
        processing_log.append({
            "file": file_name,
            "status": "failed",
            "start_time": file_start_time,
            "duration_seconds": duration,
            "rows_written": None,
            "output_file": None,
            "error": str(e)
        })

processing_log_df = pd.DataFrame(processing_log)
processing_log_df

## 6. Combine all MFCC CSVs and align with environmental variables

In [ ]:
mfcc_files = sorted(glob(os.path.join(MFCC_OUTPUT_FOLDER, "*_mfcc.csv")))

combined_mfcc_df = pd.concat(
    [pd.read_csv(f, parse_dates=["timestamp"]) for f in mfcc_files],
    ignore_index=True
)

aligned_df = align_mfcc_with_environmental_log(combined_mfcc_df, env_df)

required_columns = ["mfcc_1", "mfcc_2", "mfcc_3", "Temperatura (°C)", "Wilgotność (%)"]
available_required = [col for col in required_columns if col in aligned_df.columns]

aligned_df = aligned_df.dropna(subset=available_required).reset_index(drop=True)
aligned_df.head()

## 7. Save merged aligned dataset

In [ ]:
aligned_output_path = os.path.join(MFCC_OUTPUT_FOLDER, "mfcc_aligned_with_environment.csv")
aligned_df.to_csv(aligned_output_path, index=False)

aligned_output_path

## 8. Normalise selected MFCCs and environmental variables for comparison

Normalisation is used here **only for visual comparison**. The original MFCC values are preserved in the aligned dataset and should be used for any later statistical analysis that requires raw feature values.

In [ ]:
columns_to_normalise = PLOT_MFCCS + ["Temperatura (°C)", "Wilgotność (%)"]
columns_to_normalise = [col for col in columns_to_normalise if col in aligned_df.columns]

norm_df = normalise_columns(aligned_df, columns=columns_to_normalise)

norm_output_path = os.path.join(MFCC_OUTPUT_FOLDER, "mfcc_aligned_normalised.csv")
norm_df.to_csv(norm_output_path, index=False)

norm_df.head()

## 9. Plot full-session overview

In [ ]:
plot_mfcc_overview(
    norm_df=norm_df,
    mfcc_columns=PLOT_MFCCS,
    env_columns=["Temperatura (°C)", "Wilgotność (%)"]
)

## 10. Plot session in smaller chunks

Chunked plots make subtle temporal variation easier to inspect than a single long-duration figure. This is especially useful when looking for day-night patterns, sudden shifts, or periods that may correspond to environmental disturbance.

In [ ]:
plot_mfcc_chunks(
    norm_df=norm_df,
    mfcc_columns=PLOT_MFCCS,
    env_columns=["Temperatura (°C)", "Wilgotność (%)"],
    output_dir=PLOT_OUTPUT_FOLDER,
    chunk_seconds=CHUNK_SECONDS
)

## 11. Export processing log

In [ ]:
log_output_path = os.path.join(MFCC_OUTPUT_FOLDER, "mfcc_processing_log.csv")
processing_log_df.to_csv(log_output_path, index=False)

processing_log_df

## 12. Interpretation notes

### Why MFCCs matter in this project
MFCCs summarise the shape of the short-term frequency spectrum in a compact form. In beehive monitoring, this helps track how the spectral profile of colony sound changes over time rather than looking only at raw amplitude.

### How to interpret the lower-order coefficients
Lower-order coefficients, especially **MFCC 1-3**, often reflect broader spectral structure:
- MFCC 1 may capture large-scale changes in spectral energy distribution
- MFCC 2 and MFCC 3 may reflect shifts in dominant buzzing texture or timbral balance

### How to interpret the higher-order coefficients
Higher-order coefficients can capture finer spectral detail:
- subtle harmonic changes
- more complex texture variation
- short-term irregularities that may not be obvious in RMS alone

### Important methodological note
Because preprocessing included normalisation, MFCC changes should be discussed mainly as **relative spectral changes over time**, not direct evidence of absolute loudness increase or decrease.

### Value for the wider dissertation
MFCCs complement simpler features such as RMS and ZCR by providing richer spectral information. They are therefore useful for later:
- temporal trend analysis
- unsupervised clustering
- dimensionality reduction such as PCA
- comparison with temperature and humidity patterns